# E46 logbook

Bins, maps and RomRaider logs of the MS42 E85 tune. The logic lives in `tools/helpers.py`
(self-check: `python -m tools.check`).

In [ ]:
import pandas as pd
from IPython.display import display

from tools import (bins_overview, extract_map, fuel_trim_table, list_bins, list_logs,
                   load_log, log_browser, map_browser, to_romraider)

DEFINITION = "tools/MS42_ECU_Definitions_v0.44.xml"
bins = list_bins()   # {'stock': Path, 'E85_V0.1': Path, ...}
logs = list_logs()   # {'2022-05-26/romraiderlog_....csv': Path, ...}

## Bins

Every 32KB calibration in the repo, the bin it was based on and what changed from it.

In [ ]:
PARENTS = {"E85_V0.5": "E85_V0.3",  # V0.4 was a dead end
           "test_1": "stock"}     # default parent: the previous bin

(bins_overview(DEFINITION, bins, PARENTS)
 .style.set_properties(subset=["changed vs parent"], **{"text-align": "left", "white-space": "normal"}))

## Maps

Pick a table in **bin A** to see its values and 3D plot. Pick a **bin B** to compare: B is drawn
in red over A, and the heatmap shows B vs A in %. Tick *only tables changed* to list only what
differs between A and B (or A and stock).

In [ ]:
map_browser(DEFINITION, bins, a="E85_V0.9", b="stock",
            table="Fuel Injection part load bank 1 warm engine")

## Logs

Pick a log, the channels to plot against time (ctrl/shift-click for several; all share the zoom),
and an X/Y scatter coloured by a third channel. Added channels:
* `Injector Duty (%)` = IPW × RPM / 1200
* `Fuel Correction - Bank n (%)` = (1 + STFT)(1 + LTFT) − 1: how far the map is from what the engine needs

In [ ]:
log_browser(logs)

## Fuel trim correction per map cell

Replaces the old linear fit. Takes warm (coolant > 80°C), closed-loop points (pedal < 60%, no
overrun fuel cut), averages the fuel correction in each cell of a fuel table, and multiplies the
cell by it where there are ≥ 20 samples and the correction is ≥ 2%.

* Use logs recorded on the bin being corrected (same fuel tables).
* Narrowband sensors: the mean correction is approximate. Full load is open loop and can't be
  corrected this way.
* A uniform offset is already handled by the LTFT; only the differences between cells matter.

In [ ]:
BIN = "E85_V0.9"
TABLE = "Fuel Injection part load bank 1 warm engine"
BANK = 1
LOGS = [k for k in logs if k.startswith("2022-05-2")]  # V0.3-V0.5, same fuel tables as V0.9

fuel_map = extract_map(bins[BIN], DEFINITION, TABLE)
correction, samples, corrected = fuel_trim_table(pd.concat(load_log(logs[k]) for k in LOGS), fuel_map, bank=BANK)

display(correction.style.background_gradient(cmap="RdBu_r", vmin=-10, vmax=10, axis=None).format(precision=1, na_rep=""))
display(samples)
display(corrected.style.format(precision=3))
# import pyperclip; pyperclip.copy(to_romraider(corrected))  # then paste into the table in RomRaider